In [2]:
import GridFlowSmooth as flow
import numpy as np
import pypower
import andes
import os
import pandapower
import matplotlib.pyplot as plt
import pandas as pd

# Parallel lines

In [6]:
# see the two lines running from 1 to 2

sus = {(1,2):60, (1,2,2):140, (4,6):300, (5,7):100, (7,2):200, (1,3):200, (2,3):100, (2,4):200, (3,5):200}
line_capacity = {key:sus[key]*3 for key in sus.keys()}
poss_dc_lines = [(3,4)]
dc_lines = [(3,7)]
dc_capacity = {key:1000 for key in dc_lines + poss_dc_lines}
line_capacity.update(dc_capacity)

out, prob = flow.ACDC_TEP_OPF(gen_capacity = {1:400,2:100,3:0,4:0,5:100, 6:0, 7:0},
                susceptances = sus, line_capacity = line_capacity, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4, 5, 6, 7], obj = 'cost',
                 demands = {1: 50, 2: 60, 3: 40, 4: 30, 5:50, 6:20, 7:60}, cut = False,
                 ac_lines = [(1, 2), (1,2,2), (4,6), (5,7), (7,2)], dc_lines = dc_lines,
                 poss_ac_lines = [(1, 3),(2, 3), (2, 4), (3,5)], oldsave = False, save = False,
                 poss_dc_lines = poss_dc_lines, generation_cost = {1: 120, 2: 25, 5: 10},
                 printout = True, max_new_ac = 2, max_new_dc = 0, label="")

Status: Optimal
Objective Value: 16700.0

Generation at Each Bus:
Bus 1: 110.0 MW
Bus 2: 100.0 MW
Bus 5: 100.0 MW

AC Line Flows:
Line (1, 2): 18.0 MW
Line (1, 2, 2): 42.0 MW
Line (4, 6): 20.0 MW
Line (5, 7): 50.0 MW
Line (7, 2): -50.0 MW
New AC Line (1, 3): 0.0 MW
New AC Line (2, 3): 0.0 MW
New AC Line (2, 4): 50.0 MW
New AC Line (3, 5): 0.0 MW

DC Line Flows:
Line (3, 7): -40.0 MW

New AC Line Decisions:
Line (1, 3): Not Added
Line (2, 3): Not Added
Line (2, 4): Added
Line (3, 5): Not Added

New DC Line Decisions:
Line (3, 4): Not Added

Voltage phases:
Bus 1: -2.5249259869237295 rads
Bus 2: -2.8249259869237298 rads
Bus 3: -3.14159265359 rads
Bus 4: -3.0749259869237298 rads
Bus 5: -2.5749259869237293 rads
Bus 6: -3.14159265359 rads
Bus 7: -3.0749259869237293 rads


In [8]:
# see the single line running from 1 to 2

sus = {(1,2):300, (4,6):300, (5,7):100, (7,2):200, (1,3):200, (2,3):100, (2,4):200, (3,5):200}
line_capacity = {key:sus[key]*3 for key in sus.keys()}
poss_dc_lines = [(3,4)]
dc_lines = [(3,7)]
dc_capacity = {key:1000 for key in dc_lines + poss_dc_lines}
line_capacity.update(dc_capacity)

out, prob = flow.ACDC_TEP_OPF(gen_capacity = {1:400,2:100,3:0,4:0,5:100, 6:0, 7:0},
                susceptances = sus, line_capacity = line_capacity, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4, 5, 6, 7], obj = 'cost',
                 demands = {1: 50, 2: 60, 3: 40, 4: 30, 5:50, 6:20, 7:60}, cut = False,
                 ac_lines = [(1, 2),  (4,6), (5,7), (7,2)], dc_lines = dc_lines,
                 poss_ac_lines = [(1, 3),(2, 3), (2, 4), (3,5)], oldsave = False, save = False,
                 poss_dc_lines = poss_dc_lines, generation_cost = {1: 120, 2: 25, 5: 10},
                 printout = True, max_new_ac = 2, max_new_dc = 0, label="")

Status: Optimal
Objective Value: 16700.0

Generation at Each Bus:
Bus 1: 110.0 MW
Bus 2: 100.0 MW
Bus 5: 100.0 MW

AC Line Flows:
Line (1, 2): 60.0 MW
Line (4, 6): 20.0 MW
Line (5, 7): 50.0 MW
Line (7, 2): -50.0 MW
New AC Line (1, 3): 0.0 MW
New AC Line (2, 3): 0.0 MW
New AC Line (2, 4): 50.0 MW
New AC Line (3, 5): 0.0 MW

DC Line Flows:
Line (3, 7): -40.0 MW

New AC Line Decisions:
Line (1, 3): Not Added
Line (2, 3): Not Added
Line (2, 4): Added
Line (3, 5): Not Added

New DC Line Decisions:
Line (3, 4): Not Added

Voltage phases:
Bus 1: -2.6249259869237296 rads
Bus 2: -2.8249259869237298 rads
Bus 3: -3.14159265359 rads
Bus 4: -3.0749259869237298 rads
Bus 5: -2.5749259869237293 rads
Bus 6: -3.14159265359 rads
Bus 7: -3.0749259869237293 rads


# We strain AC, DC reigns supreme

$Flow_{i,j} = B_{i,j}(\theta_i - \theta_j)$

$\therefore  B_{i,j}= Flow_{i,j}/(\theta_i - \theta_j)$ 

so we go with

$Flow_{cumulative}/(2\pi) = B_{uniform}$

such that if a acyclical path requires a flow of 

$$Flow_{cumulative } = \sum_{subpath\in path} \sum_{link\in subpath} Flow_{link}$$

$$Flow_{cumulative } = \sum_{a =1}^N \sum_{b = a}^N Flow_{b}$$

would need at least one line with susceptance $B_{uniform}$. Usually this will be by the generator.

and our minimal susceptance is specially sensitive to the loads farthest from the generator

In [13]:
ac_lines = [(1, 2), (2, 3), (3,4), (4,5), (5,6)]
poss_ac_lines = [ (2, 4), (3,5), (6,7)]
poss_dc_lines, dc_lines = [], []
cumu_flow = 260 + 200 +160+130+80+60
sus = {key:cumu_flow/(2*np.pi) for key in ac_lines+poss_ac_lines}
line_capacity = {key:sus[key]*3 for key in ac_lines+poss_ac_lines+dc_lines + poss_dc_lines}
dc_capacity = {key:1000 for key in dc_lines + poss_dc_lines}
line_capacity.update(dc_capacity)

out, prob = flow.ACDC_TEP_OPF(gen_capacity = {1:400,2:0,3:0,4:0,5:0, 6:0, 7:0},
                susceptances = sus, line_capacity = line_capacity, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4, 5, 6, 7], obj = 'cost',
                 demands = {1: 50, 2: 60, 3: 40, 4: 30, 5:50, 6:20, 7:60}, cut = False,
                 ac_lines = ac_lines, dc_lines = dc_lines,
                 poss_ac_lines = poss_ac_lines, oldsave = False, save = False,
                 poss_dc_lines = poss_dc_lines, generation_cost = {1: 120, 2: 25, 5: 10},
                 printout = True, max_new_ac = 1, max_new_dc = 0, label="")

Status: Optimal
Objective Value: 37200.0

Generation at Each Bus:
Bus 1: 310.0 MW

AC Line Flows:
Line (1, 2): 260.0 MW
Line (2, 3): 200.0 MW
Line (3, 4): 160.0 MW
Line (4, 5): 130.0 MW
Line (5, 6): 80.0 MW
New AC Line (2, 4): 0.0 MW
New AC Line (3, 5): 0.0 MW
New AC Line (6, 7): 60.0 MW

New AC Line Decisions:
Line (2, 4): Not Added
Line (3, 5): Not Added
Line (6, 7): Added

New DC Line Decisions:

Voltage phases:
Bus 1: 3.1415926535838166 rads
Bus 2: 1.306055372837486 rads
Bus 3: -0.10589638158276848 rads
Bus 4: -1.2354577851189719 rads
Bus 5: -2.153226425492137 rads
Bus 6: -2.718007127260239 rads
Bus 7: -3.141592653588138 rads


the network above is strained to capacity. 

the network below only survives by adding (6,7) dc line. adding (6,7) ac line won't help.

In [27]:
ac_lines = [(1, 2), (2, 3), (3,4), (4,5), (5,6)]
poss_ac_lines = [ (2, 4), (3,5), (6,7)]
poss_dc_lines, dc_lines = [(6,7)], []
cumu_flow = 260 + 200 +160+130+80+60
sus = {key:cumu_flow/(2*np.pi) for key in ac_lines+poss_ac_lines}
line_capacity = {key:sus[key]*3 for key in ac_lines+poss_ac_lines+dc_lines + poss_dc_lines}
dc_capacity = {key:1000 for key in dc_lines + poss_dc_lines}
line_capacity.update(dc_capacity)

out, prob = flow.ACDC_TEP_OPF(gen_capacity = {1:400,2:0,3:0,4:0,5:0, 6:0, 7:0},
                susceptances = sus, line_capacity = line_capacity, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4, 5, 6, 7], obj = 'cost',
                 demands = {1: 51, 2: 61, 3: 41, 4: 31, 5:51, 6:21, 7:61}, cut = False,
                 ac_lines = ac_lines, dc_lines = dc_lines,
                 poss_ac_lines = poss_ac_lines, oldsave = False, save = False,
                 poss_dc_lines = poss_dc_lines, generation_cost = {1: 120, 2: 25, 5: 10},
                 printout = True, max_new_ac = 1, max_new_dc = 1, label="")

Status: Optimal
Objective Value: 38040.0

Generation at Each Bus:
Bus 1: 317.0 MW

AC Line Flows:
Line (1, 2): 266.0 MW
Line (2, 3): 205.0 MW
Line (3, 4): 164.0 MW
Line (4, 5): 133.0 MW
Line (5, 6): 82.0 MW
New AC Line (2, 4): 0.0 MW
New AC Line (3, 5): 0.0 MW
New AC Line (6, 7): 0.0 MW
New DC Line (6, 7): 61.0 MW

New AC Line Decisions:
Line (2, 4): Not Added
Line (3, 5): Not Added
Line (6, 7): Not Added

New DC Line Decisions:
Line (6, 7): Added

Voltage phases:
Bus 1: 2.8592023026960676 rads
Bus 2: 0.9813064693171293 rads
Bus 3: -0.4659440789636314 rads
Bus 4: -1.6237445175882383 rads
Bus 5: -2.5626924342777073 rads
Bus 6: -3.14159265359 rads
Bus 7: -3.14159265359 rads


# Loops

In [ ]:
ac_lines = [(1, 2), (2, 3), (3,4), (4,5), (5,2)]
poss_ac_lines = [ (2, 4), (3,5), (6,7)]
poss_dc_lines, dc_lines = [], []
cumu_flow = 260 + 200 +160+130+80+60
sus = {key:cumu_flow/(2*np.pi) for key in ac_lines+poss_ac_lines}
line_capacity = {key:sus[key]*3 for key in ac_lines+poss_ac_lines+dc_lines + poss_dc_lines}
dc_capacity = {key:1000 for key in dc_lines + poss_dc_lines}
line_capacity.update(dc_capacity)

out, prob = flow.ACDC_TEP_OPF(gen_capacity = {1:400,2:0,3:0,4:0,5:0, 6:0, 7:0},
                susceptances = sus, line_capacity = line_capacity, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4, 5, 6, 7], obj = 'cost',
                 demands = {1: 50, 2: 60, 3: 40, 4: 30, 5:50, 6:20, 7:60}, cut = False,
                 ac_lines = ac_lines, dc_lines = dc_lines,
                 poss_ac_lines = poss_ac_lines, oldsave = False, save = False,
                 poss_dc_lines = poss_dc_lines, generation_cost = {1: 120, 2: 25, 5: 10},
                 printout = True, max_new_ac = 1, max_new_dc = 0, label="")

# Multiple Generators

For a node with two transmission connections to generators to fail via susceptance, we need:

max flow delivered from both lines to undercut the demand

In [66]:
ac_lines = [(1, 2), (2, 3), (3,4), (4,5), ]
poss_ac_lines = [ (2, 4), (3,5),]
poss_dc_lines, dc_lines = [], []
cum_flow = 50*4+60*3+40*2+30
# straining the network if gen 5 was alone
sus = {key:cum_flow/(2*np.pi) for key in ac_lines+poss_ac_lines}
line_capacity = {key:sus[key]*3 for key in ac_lines+poss_ac_lines+dc_lines + poss_dc_lines}
dc_capacity = {key:1000 for key in dc_lines + poss_dc_lines}
line_capacity.update(dc_capacity)

out, prob = flow.ACDC_TEP_OPF(gen_capacity = {1:400,2:0,3:0,4:0,5:400},
                susceptances = sus, line_capacity = line_capacity, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4, 5], obj = 'cost',
                 demands = {1: 50, 2: 60, 3: 40, 4: 30, 5:50}, cut = False,
                 ac_lines = ac_lines, dc_lines = dc_lines,
                 poss_ac_lines = poss_ac_lines, oldsave = False, save = False,
                 poss_dc_lines = poss_dc_lines, generation_cost = {1: 120, 2: 25, 5: 10},
                 printout = True, max_new_ac = 0, max_new_dc = 0, label="")

Status: Optimal
Objective Value: 2300.0

Generation at Each Bus:
Bus 5: 230.0 MW

AC Line Flows:
Line (1, 2): -50.0 MW
Line (2, 3): -110.0 MW
Line (3, 4): -150.0 MW
Line (4, 5): -180.0 MW

Voltage phases:
Bus 1: -3.14159265359 rads
Bus 2: -2.500451295714282 rads
Bus 3: -1.0899403083877195 rads
Bus 4: 0.8334837652394063 rads
Bus 5: 3.14159265359 rads


In [54]:
1/np.pi*180
#we'd like less than half radian angle differences...

57.29577951308232

In [68]:
ac_lines = [(1, 2), (2, 3), (3,4), (4,5), ]
poss_ac_lines = [ (2, 4), (3,5),]
poss_dc_lines, dc_lines = [], []
cum_flow = 60*3+40*2+30
# straining the network so gen 5 supplies all but bus 1 which handles own load
sus = {key:cum_flow/(2*np.pi) for key in ac_lines+poss_ac_lines}
line_capacity = {key:sus[key]*3 for key in ac_lines+poss_ac_lines+dc_lines + poss_dc_lines}
dc_capacity = {key:1000 for key in dc_lines + poss_dc_lines}
line_capacity.update(dc_capacity)

out, prob = flow.ACDC_TEP_OPF(gen_capacity = {1:400,2:0,3:0,4:0,5:400},
                susceptances = sus, line_capacity = line_capacity, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4, 5], obj = 'cost',
                 demands = {1: 50, 2: 60, 3: 40, 4: 30, 5:50}, cut = False,
                 ac_lines = ac_lines, dc_lines = dc_lines,
                 poss_ac_lines = poss_ac_lines, oldsave = False, save = False,
                 poss_dc_lines = poss_dc_lines, generation_cost = {1: 120, 2: 25, 5: 10},
                 printout = True, max_new_ac = 0, max_new_dc = 0, label="")

Status: Optimal
Objective Value: 7800.0

Generation at Each Bus:
Bus 1: 50.0 MW
Bus 5: 180.0 MW

AC Line Flows:
Line (1, 2): 0.0 MW
Line (2, 3): -60.0 MW
Line (3, 4): -100.0 MW
Line (4, 5): -130.0 MW

Voltage phases:
Bus 1: -3.141592653587885 rads
Bus 2: -3.14159265359 rads
Bus 3: -1.8416232796914918 rads
Bus 4: 0.32499234347409844 rads
Bus 5: 3.14159265359 rads


In [9]:
ac_lines = [(1, 2), (2, 3), (3,4), (4,5), ]
poss_ac_lines = [ (2, 4), (3,5),]
poss_dc_lines, dc_lines = [], []
cum_flow = 110
# gen 5 would need to send 40*2 + 30 to supply 3, 
# gen 1 would need to send 40*2 + 60 to supply 3,
# gen 5 is less strained. 
# if we substitute that 40 for a 20, each contirbuting half

sus = {key:cum_flow/(2*np.pi) for key in ac_lines+poss_ac_lines}
line_capacity = {key:sus[key]*3 for key in ac_lines+poss_ac_lines+dc_lines + poss_dc_lines}
dc_capacity = {key:1000 for key in dc_lines + poss_dc_lines}
line_capacity.update(dc_capacity)

out, prob = flow.ACDC_TEP_OPF(gen_capacity = {1:400,2:0,3:0,4:0,5:400},
                susceptances = sus, line_capacity = line_capacity, 
                 line_expansion_cost = 1000, buses = [1, 2, 3, 4, 5], obj = 'cost',
                 demands = {1: 50, 2: 60, 3: 40, 4: 30, 5:50}, cut = False,
                 ac_lines = ac_lines, dc_lines = dc_lines,
                 poss_ac_lines = poss_ac_lines, oldsave = False, save = False,
                 poss_dc_lines = poss_dc_lines, generation_cost = {1: 120, 2: 25, 5: 10},
                 printout = True, max_new_ac = 0, max_new_dc = 0, label="")

Did not converge
See returned problem object


NameError: name 'sns' is not defined